### **Table of Content**
- Chapter 1.0: Running Hugging Face Model

### **Key Highlights**
-
-
-

### **Website**
https://huggingface.co/

In [2]:
from transformers import pipeline
import os
from huggingface_hub import InferenceClient
from dotenv import load_dotenv, find_dotenv
from datasets import load_dataset

In [2]:
# find_dotenv() will automatically climb up from 'developer_path' 
# to the root folder to find your .env file flawlessly.
load_dotenv(find_dotenv())

# Retrieve the key
api_key = os.getenv("HUGGING_FACE_API_KEY")

# Safety check to make sure it loaded
if not api_key:
    raise ValueError("API Key is still missing! Double-check the variable name inside your .env file.")

# Initialize the client
client = InferenceClient(token=api_key)
print("Connected successfully! Hugging Face Inference client is ready.")

Connected successfully! Hugging Face Inference client is ready.


#### **1.0 Running Hugging Face Model**


In [ ]:
# build the example pipeline
gpt2_pipeline = pipeline(

    task="text-generation",
    model="openai-community/gpt2"

)

# test the pipeline
print(gpt2_pipeline("What if AI"))

In [ ]:
client = InferenceClient(
    provider="together",
    api_key=os.getenv("TOGETHER_API_KEY")
)

#### **2.0 Downloading a dataset**


In [ ]:
dataset = load_dataset("IVN-RIN/BioBERT_Italian", split="train")
filtered_dataset = dataset.filter(lambda row: " bella " in row["text"])
print(filtered_dataset)


In [ ]:
sliced = filtered_dataset.select(range(5))
print(sliced)
print(sliced[0]["text"])

#### **3.0 Text Classification**


In [ ]:
# sentiment analysis pipeline
pipeline = pipeline(
    task="text-classification", # specify the task
    model="distilbert-base-uncased-finetuned-sst-2-english" # from HF hub
)

print(pipeline("This is a great movie!"))

In [ ]:
# grammatical error correction pipeline
pipeline = pipeline(
    task="text2text-generation", # specify the task
    model="facebook/bart-large-cnn" # from HF hub
)
print(pipeline("She no went to the market yesterday."))

In [ ]:
# qnli
pipeline = pipeline(
    task="question-answering",
    model="cross-encoder/quora-distilroberta-base" # from HF hub
)
print(pipeline("What is the capital of France?", "Paris"))

In [ ]:
# dynamic category
pipeline = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli" # from HF hub
)

text = "I love programming in Python!"
cats = ["technology", "sports", "politics"]

output = pipeline(text, cats)
print(f"Top Label: {output['labels'][0]}, Score: {output['scores'][0]:.4f}")

#### **4.0 Text Summarization**
- Extractive: Key Sentences, Efficient, Fewer Resources
- Abstractive: Generate/Rephrased Text, Clear, More readable


In [ ]:
# extractive
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn" # from HF hub
)

text = "This is a very long sentence that needs to be summarized. It contains a lot of information and details that might not be necessary for the main point."
summarized = summarizer(text)

print(summarized[0]['summary_text'])

In [ ]:
# abstractive
summarizer = pipeline(
    task="summarization",
    model="google/pegasus-xsum", # from HF hub
    min_new_tokens=5,
    max_new_tokens=20
)

text = "This is a very long sentence that needs to be summarized. It contains a lot of information and details that might not be necessary for the main point."
summarized = summarizer(text)
print(summarized[0]['summary_text'])

#### **5.0 Auto Models & Tokenizers**
- Auto classes, more control & perfect for advanced tasks


In [1]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

In [3]:
text = "I love this product!"

In [4]:
# load the model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\NAZMIRULIZZADBINNASI\anaconda3\envs\datacamp_venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NAZMIRULIZZADBINNASI\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [5]:
# load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
tokens = tokenizer.tokenize(text)
print(tokens)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

['i', 'love', 'this', 'product', '!']


In [ ]:
pipeline = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
# full customize pipeline
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

pipeline = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

#### **6.0 Document Q&A**
- Answering question/inquiries based on document

In [7]:
from pypdf import PdfReader

In [ ]:
reader = PdfReader("example.pdf")
page = reader.pages[0]
text = page.extract_text()
print(text)

# looping through all pages
for page in reader.pages:
    text = page.extract_text()
    print(text)

In [ ]:
qa_pipeline = pipeline(
    task="question-answering",
    model="cross-encoder/quora-distilroberta-base" # from HF hub
)

question = "What is the duration needed to pay the tuition fee?"
result = qa_pipeline(question, text)
print(f"Answer: {result['answer']}, Score: {result['score']:.4f}")